2.1 理论计算题

卷积层输出特征图尺寸计算
卷积输出尺寸公式：
\(H_{out} = \lfloor \frac{H_{in} + 2 \times padding - kernel\_size}{stride} \rfloor + 1\)
\(W_{out} = \lfloor \frac{W_{in} + 2 \times padding - kernel\_size}{stride} \rfloor + 1\)
已知输入尺寸为 \(3 \times 32 \times 32\)（通道 × 高 × 宽），卷积核大小 \(5 \times 5\)，padding=2，stride=2，卷积核数量 16：

输出通道数 = 卷积核数量 = 16
输出高度：\(\lfloor \frac{32 + 2 \times 2 - 5}{2} \rfloor + 1 = \lfloor \frac{31}{2} \rfloor + 1 = 15 + 1 = 16\)
输出宽度：与高度相同，为 16
最终输出特征图尺寸：\(\boldsymbol{16 \times 16 \times 16}\)



单个输出通道单个像素的点乘次数
单个卷积核的维度为 \(3 \times 5 \times 5\)（输入通道 × 核高 × 核宽），每个输出像素由卷积核与输入对应区域逐元素相乘后求和得到，乘法次数等于卷积核的元素总数：
\(3 \times 5 \times 5 = \boldsymbol{75}\)

In [6]:
import numpy as np

def max_pool2d_forward(x, kernel_size, stride=1, padding=0):
    """
    手动实现二维最大池化前向传播
    参数:
        x: 输入张量，形状为 (N, C, H, W) (批量数, 通道数, 高度, 宽度)
        kernel_size: 池化核大小，int或tuple (kh, kw)
        stride: 步幅，int或tuple (sh, sw)
        padding: 填充，int或tuple (ph, pw)
    返回:
        out: 池化后的输出张量
    """
    # 统一参数格式
    if isinstance(kernel_size, int):
        kh, kw = kernel_size, kernel_size
    else:
        kh, kw = kernel_size
    
    if isinstance(stride, int):
        sh, sw = stride, stride
    else:
        sh, sw = stride
    
    if isinstance(padding, int):
        ph, pw = padding, padding
    else:
        ph, pw = padding
    
    N, C, H, W = x.shape
    
    # 计算输出尺寸
    H_out = (H + 2 * ph - kh) // sh + 1
    W_out = (W + 2 * pw - kw) // sw + 1
    
    # 对输入进行零填充
    x_pad = np.pad(x, ((0,0), (0,0), (ph, ph), (pw, pw)), mode='constant', constant_values=0)
    
    # 初始化输出
    out = np.zeros((N, C, H_out, W_out))
    
    # 遍历每个批量、每个通道、每个输出位置
    for n in range(N):
        for c in range(C):
            for i in range(H_out):
                for j in range(W_out):
                    # 计算池化窗口在输入上的位置
                    h_start = i * sh
                    h_end = h_start + kh
                    w_start = j * sw
                    w_end = w_start + kw
                    # 取窗口内的最大值
                    out[n, c, i, j] = np.max(x_pad[n, c, h_start:h_end, w_start:w_end])
    
    return out

# 测试示例
if __name__ == "__main__":
    x = np.random.randn(1, 1, 4, 4)  # 1个批量，1个通道，4x4输入
    print("输入：")
    print(x[0,0])
    out = max_pool2d_forward(x, kernel_size=2, stride=2, padding=0)
    print("\n2x2最大池化（步幅2）输出：")
    print(out[0,0])

输入：
[[ 0.68721561 -0.16496231  0.54990004 -0.6583126 ]
 [ 0.17765673  0.68633716 -1.77760136  0.86599903]
 [-0.09440229  1.17817923 -0.44238135  1.38585028]
 [ 0.71014819 -0.6683371   0.01511608 -1.53136665]]

2x2最大池化（步幅2）输出：
[[0.68721561 0.86599903]
 [1.17817923 1.38585028]]


3.1 理论计算题
单个 5×5 卷积层（不带偏置）的参数量
卷积层参数量计算公式（无偏置项）：
参数量 = 输入通道数 × 输出通道数 × 卷积核高度 × 卷积核宽度
题目中输入和输出通道数均为 C，卷积核大小为 5×5，代入得：
C × C × 5 × 5 = 25C²
两个串联 3×3 卷积层（不带偏置，两层通道数都为 C）的总参数量
单个 3×3 卷积层（无偏置）的参数量为：
C × C × 3 × 3 = 9C²
两个完全相同的 3×3 卷积层串联，总参数量为两层之和：
9C² + 9C² = 18C²

In [7]:
import torch
import torch.nn as nn

def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """
    定义标准NiN块
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        kernel_size: 第一个卷积层的核大小
        stride: 第一个卷积层的步幅
        padding: 第一个卷积层的填充
    返回:
        nn.Sequential: NiN块
    """
    return nn.Sequential(
        # 第一个普通卷积层 + ReLU
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(),
        # 第一个1×1卷积层 + ReLU
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        # 第二个1×1卷积层 + ReLU
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU()
    )

# 测试示例
if __name__ == "__main__":
    block = nin_block(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=2)
    x = torch.randn(1, 3, 32, 32)
    out = block(x)
    print("NiN块输出形状:", out.shape)  # 应输出 torch.Size([1, 16, 32, 32])

NiN块输出形状: torch.Size([1, 16, 32, 32])


4.1 理论计算题：批量归一化计算
批量归一化完整计算步骤：
计算批量均值：μ_B = 所有样本值的平均值
计算批量方差：σ_B² = 所有样本值与均值差的平方的平均值
归一化：x̂_i = (x_i - μ_B) / √(σ_B² + ε)
缩放平移：y_i = γ × x̂_i + β
已知条件：
批量样本值：x₁=2, x₂=4, x₃=6, x₄=8
缩放参数：γ=2
平移参数：β=1
稳定常数：ε=0
计算过程：
批量均值：μ_B = (2+4+6+8) ÷ 4 = 5
批量方差：σ_B² = [(2-5)² + (4-5)² + (6-5)² + (8-5)²] ÷ 4 = (9+1+1+9) ÷ 4 = 5
归一化值：
x̂₁ = (2-5) ÷ √5 ≈ -1.3416
x̂₂ = (4-5) ÷ √5 ≈ -0.4472
x̂₃ = (6-5) ÷ √5 ≈ 0.4472
x̂₄ = (8-5) ÷ √5 ≈ 1.3416
最终输出：
y₁ = 2 × (-1.3416) + 1 ≈ -1.683
y₂ = 2 × (-0.4472) + 1 ≈ 0.106
y₃ = 2 × 0.4472 + 1 ≈ 1.894
y₄ = 2 × 1.3416 + 1 ≈ 3.683

In [8]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    自定义残差块类
    """
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        # 第一个3×3卷积层 + BN + ReLU
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        
        # 第二个3×3卷积层 + BN
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 1×1卷积用于调整输入的通道数和尺寸
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.conv3 = None
    
    def forward(self, x):
        # 主路径
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # 残差路径
        if self.conv3:
            x = self.conv3(x)
        
        # 按元素相加后激活
        out += x
        out = self.relu(out)
        return out

# 测试示例
if __name__ == "__main__":
    # 测试通道数不变的情况
    blk1 = Residual(3, 3)
    x = torch.randn(1, 3, 32, 32)
    out1 = blk1(x)
    print("通道数不变的残差块输出形状:", out1.shape)  # 应输出 torch.Size([1, 3, 32, 32])
    
    # 测试通道数和尺寸变化的情况
    blk2 = Residual(3, 16, use_1x1conv=True, stride=2)
    out2 = blk2(x)
    print("通道数和尺寸变化的残差块输出形状:", out2.shape)  # 应输出 torch.Size([1, 16, 16, 16])

通道数不变的残差块输出形状: torch.Size([1, 3, 32, 32])
通道数和尺寸变化的残差块输出形状: torch.Size([1, 16, 16, 16])


5.1 理论计算题
底层特征提取层学习率小、顶层输出层学习率大的原因
底层特征的通用性：底层卷积层学习的是边缘、纹理、颜色等通用视觉特征，这些特征在源数据集（如 ImageNet）上已经训练得非常完善，且适用于大多数计算机视觉任务。小学习率可以保留这些已学到的有用知识，避免大幅修改破坏通用特征。
顶层特征的任务特异性：顶层输出层是针对源数据集的特定分类任务设计的，其学习到的特征与源数据集的类别高度相关，无法直接迁移到目标任务。且顶层通常是随机初始化的，需要较大的学习率来快速学习目标数据集的特定模式。
参数数量与过拟合风险：底层特征提取层包含大量参数，使用大学习率容易导致参数震荡和过拟合；顶层参数数量少，大学习率训练更稳定且收敛更快。
目标数据集很小且与源数据集相似时的微调策略
冻结全部底层特征提取层：只训练最后一层或几层全连接输出层，大幅减少需要训练的参数数量，从根本上降低过拟合风险。
增强图像增广强度：通过随机裁剪、翻转、色彩抖动、旋转等方式扩充训练数据，增加数据多样性，提高模型泛化能力。
使用小批量和低学习率：采用较小的批量大小（如 8-16）和较低的学习率（如 1e-4 到 1e-3）训练顶层，避免参数更新过快导致过拟合。
加入正则化手段：在全连接层中使用 Dropout，同时添加权重衰减（L2 正则化），进一步抑制过拟合。
早停策略：监控验证集性能，当验证集准确率不再提升甚至下降时立即停止训练，防止模型在训练集上过拟合。

In [9]:
from torchvision import transforms

def get_augmentation_pipeline():
    """
    创建符合要求的图像增广管道
    """
    return transforms.Compose([
        # 1. 随机裁剪并缩放到224×224，面积比例0.08-1.0
        transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
        # 2. 50%概率水平翻转
        transforms.RandomHorizontalFlip(p=0.5),
        # 3. 随机改变亮度、对比度、饱和度，变化范围0.5
        transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
        # 4. 转换为PyTorch张量
        transforms.ToTensor()
    ])

# 测试示例
if __name__ == "__main__":
    from PIL import Image
    import numpy as np
    
    # 创建随机测试图像
    img = Image.fromarray(np.random.randint(0, 255, (500, 500, 3), dtype=np.uint8))
    aug = get_augmentation_pipeline()
    augmented_img = aug(img)
    print("增广后图像形状:", augmented_img.shape)  # 应输出 torch.Size([3, 224, 224])

增广后图像形状: torch.Size([3, 224, 224])


6.1 理论计算题：IoU 计算交并比公式：
\(IoU = \frac{\text{交集面积}}{\text{并集面积}} = \frac{A \cap B}{A \cup B}\)已知真实框 \(A=[10,10,50,50]\)，预测框 \(B=[30,30,70,70]\)：

计算两个框的面积：

A 的宽度：\(50-10=40\)，高度：\(50-10=40\)，面积 \(A=40 \times 40=1600\)
B 的宽度：\(70-30=40\)，高度：\(70-30=40\)，面积 \(B=40 \times 40=1600\)



计算交集区域：

交集左上角坐标：\((\max(10,30), \max(10,30))=(30,30)\)
交集右下角坐标：\((\min(50,70), \min(50,70))=(50,50)\)
交集宽度：\(50-30=20\)，高度：\(50-30=20\)，交集面积 \(=20 \times 20=400\)



计算并集面积：
\(A \cup B = 1600 + 1600 - 400 = 2800\)


计算 IoU：
\(IoU = \frac{400}{2800} = \boldsymbol{\frac{1}{7} \approx 0.1429}\)

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, targets, epsilon=0.1):
    """
    计算标签平滑后的交叉熵损失
    参数:
        logits: 模型输出的未归一化预测值，形状为 (N, K) (批量数, 类别数)
        targets: 真实标签，形状为 (N,)，取值范围 [0, K-1]
        epsilon: 平滑因子
    返回:
        loss: 平均损失值
    """
    K = logits.size(1)  # 类别数
    # 计算标签平滑后的目标概率
    one_hot = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1.0)
    smoothed_targets = (1 - epsilon) * one_hot + epsilon / (K - 1) * (1 - one_hot)
    
    # 计算交叉熵损失
    log_probs = F.log_softmax(logits, dim=1)
    loss = -torch.sum(smoothed_targets * log_probs, dim=1)
    
    # 返回平均损失
    return loss.mean()

# 测试示例
if __name__ == "__main__":
    logits = torch.randn(4, 5)  # 4个样本，5分类
    targets = torch.tensor([0, 1, 2, 3])  # 真实标签
    loss = label_smoothing_cross_entropy(logits, targets, epsilon=0.1)
    print("标签平滑交叉熵损失:", loss.item())

标签平滑交叉熵损失: 1.7106231451034546
